<a href="https://colab.research.google.com/github/ayyanarh1/tamil-nadu-school-flood-risk/blob/main/day10_decision_report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Setup
!pip install openpyxl fpdf2 -q

import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import (
    PatternFill, Font, Alignment,
    Border, Side
)
from openpyxl.utils import get_column_letter

print('✅ Day 10 ready!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 10.4 MB/s eta 0:00:00
✅ Day 10 ready!


In [3]:
# Master dataset with all risk components
import pandas as pd
import numpy as np

master = {
    'rank': list(range(1, 16)),
    'school_name': [
        'School Puducherry Border',
        'Panchayat School Nagapattinam',
        'School Ramanathapuram',
        'School Kanchipuram',
        'Panchayat School Tirunelveli',
        'Govt School Cuddalore',
        'School Villupuram',
        'School Tuticorin',
        'Govt High School Chennai',
        'Govt School Tiruchirappalli',
        'Govt School Thanjavur',
        'High School Vellore',
        'Govt School Salem',
        'Govt School Madurai',
        'High School Coimbatore'
    ],
    'district': [
        'Villupuram', 'Nagapattinam', 'Ramanathapuram',
        'Kanchipuram', 'Tirunelveli', 'Cuddalore',
        'Villupuram', 'Thoothukudi', 'Chennai',
        'Tiruchirappalli', 'Thanjavur', 'Vellore',
        'Salem', 'Madurai', 'Coimbatore'
    ],
    'connectivity': [
        'Connected', 'No connectivity', 'No connectivity',
        'No connectivity', 'No connectivity', 'No connectivity',
        'No connectivity', 'Connected', 'Connected',
        'No connectivity', 'Connected', 'Connected',
        'Connected', 'Connected', 'Connected'
    ],
    'hev_score': [
        68.7, 68.3, 67.0, 66.6, 65.9,
        65.6, 62.3, 54.6, 44.8, 41.1,
        33.7, 29.5, 24.7, 23.3, 18.7
    ],
    'hev_risk': [
        'CRITICAL', 'CRITICAL', 'CRITICAL', 'CRITICAL', 'CRITICAL',
        'CRITICAL', 'CRITICAL', 'CRITICAL', 'CRITICAL', 'CRITICAL',
        'HIGH', 'HIGH', 'HIGH', 'HIGH', 'MEDIUM'
    ],
    'flood_score': [
        96.7, 56.8, 43.5, 33.3, 30.7,
        39.8, 22.9, 39.1, 13.8, 31.4,
        26.5, 22.1, 4.9, 13.0, 4.8
    ],
    'cyclone_score': [
        87.5, 79.2, 45.8, 79.2, 37.5,
        87.5, 83.3, 37.5, 100.0, 50.0,
        58.3, 66.7, 62.5, 41.7, 25.0
    ],
    'vulnerability': [
        19.9, 55.0, 79.2, 68.2, 90.0,
        55.0, 59.8, 45.7, 0.1, 40.2,
        15.0, 0.1, 0.1, 0.1, 0.0
    ],
    'hospital_km': [
        19.1, 0.1, 94.1, 51.5, 136.1,
        0.2, 18.8, 119.3, 0.3, 0.7,
        0.2, 0.6, 0.5, 0.6, 0.2
    ],
    'risk_2050_ssp585': [
        'CRITICAL', 'CRITICAL', 'CRITICAL', 'CRITICAL', 'CRITICAL',
        'CRITICAL', 'CRITICAL', 'CRITICAL', 'CRITICAL', 'CRITICAL',
        'HIGH', 'HIGH', 'HIGH', 'HIGH', 'MEDIUM'
    ]
}

df = pd.DataFrame(master)

# Add recommended action
def get_action(row):
    if row['hev_risk'] == 'CRITICAL' and row['connectivity'] == 'No connectivity':
        return 'URGENT: Connectivity + flood resilience investment'
    elif row['hev_risk'] == 'CRITICAL' and row['hospital_km'] > 50:
        return 'URGENT: Emergency access route + flood shelter'
    elif row['hev_risk'] == 'CRITICAL':
        return 'HIGH PRIORITY: Flood resilience planning'
    elif row['hev_risk'] == 'HIGH':
        return 'MONITOR: Include in disaster preparedness plan'
    else:
        return 'ROUTINE: Standard monitoring'

df['recommended_action'] = df.apply(get_action, axis=1)

print(f'✅ Master dataset ready — {len(df)} schools')
print(df[['rank', 'school_name', 'hev_risk',
          'recommended_action']].to_string(index=False))

✅ Master dataset ready — 15 schools
 rank                   school_name hev_risk                                 recommended_action
    1      School Puducherry Border CRITICAL           HIGH PRIORITY: Flood resilience planning
    2 Panchayat School Nagapattinam CRITICAL URGENT: Connectivity + flood resilience investment
    3         School Ramanathapuram CRITICAL URGENT: Connectivity + flood resilience investment
    4            School Kanchipuram CRITICAL URGENT: Connectivity + flood resilience investment
    5  Panchayat School Tirunelveli CRITICAL URGENT: Connectivity + flood resilience investment
    6         Govt School Cuddalore CRITICAL URGENT: Connectivity + flood resilience investment
    7             School Villupuram CRITICAL URGENT: Connectivity + flood resilience investment
    8              School Tuticorin CRITICAL     URGENT: Emergency access route + flood shelter
    9      Govt High School Chennai CRITICAL           HIGH PRIORITY: Flood resilience planning
   1

In [4]:
# Build Excel decision report
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

wb = Workbook()

# ── Sheet 1 — Priority Action List ──
ws1 = wb.active
ws1.title = 'Priority Action List'

# Colors
RED    = PatternFill('solid', fgColor='CC0000')
ORANGE = PatternFill('solid', fgColor='FF6600')
YELLOW = PatternFill('solid', fgColor='FFAA00')
GREEN  = PatternFill('solid', fgColor='2D8A4E')
BLUE   = PatternFill('solid', fgColor='065A82')
GRAY   = PatternFill('solid', fgColor='F0F0F0')
WHITE  = PatternFill('solid', fgColor='FFFFFF')

# Fonts
bold_white  = Font(bold=True, color='FFFFFF', size=11)
bold_black  = Font(bold=True, color='000000', size=11)
normal      = Font(color='000000', size=10)
title_font  = Font(bold=True, color='FFFFFF', size=14)

# Title row
ws1.merge_cells('A1:H1')
ws1['A1'] = 'Tamil Nadu School Flood Risk — Priority Action List'
ws1['A1'].fill = BLUE
ws1['A1'].font = title_font
ws1['A1'].alignment = Alignment(horizontal='center', vertical='center')
ws1.row_dimensions[1].height = 30

# Subtitle
ws1.merge_cells('A2:H2')
ws1['A2'] = 'H x E x V Risk Framework | Sources: Sentinel-1, JRC, GFD, ERA5, IBTrACS | March 2026'
ws1['A2'].fill = PatternFill('solid', fgColor='E6F1FB')
ws1['A2'].font = Font(italic=True, size=9, color='065A82')
ws1['A2'].alignment = Alignment(horizontal='center')
ws1.row_dimensions[2].height = 18

# Headers
headers = [
    'Rank', 'School Name', 'District', 'Connectivity',
    'HEV Score', 'Risk Tier', '2050 Risk', 'Recommended Action'
]
for col, header in enumerate(headers, 1):
    cell = ws1.cell(row=3, column=col, value=header)
    cell.fill = BLUE
    cell.font = bold_white
    cell.alignment = Alignment(horizontal='center', vertical='center')
ws1.row_dimensions[3].height = 22

# Risk tier colors
tier_fills = {
    'CRITICAL': PatternFill('solid', fgColor='FFCCCC'),
    'HIGH':     PatternFill('solid', fgColor='FFE5CC'),
    'MEDIUM':   PatternFill('solid', fgColor='FFF5CC'),
    'LOW':      PatternFill('solid', fgColor='CCEECC'),
}
tier_fonts = {
    'CRITICAL': Font(bold=True, color='CC0000', size=10),
    'HIGH':     Font(bold=True, color='FF6600', size=10),
    'MEDIUM':   Font(bold=True, color='CC8800', size=10),
    'LOW':      Font(bold=True, color='2D8A4E', size=10),
}

# Data rows
for i, row in df.iterrows():
    excel_row = i + 4
    row_fill = GRAY if i % 2 == 0 else WHITE

    values = [
        row['rank'],
        row['school_name'],
        row['district'],
        row['connectivity'],
        row['hev_score'],
        row['hev_risk'],
        row['risk_2050_ssp585'],
        row['recommended_action']
    ]

    for col, value in enumerate(values, 1):
        cell = ws1.cell(row=excel_row, column=col, value=value)
        cell.fill = row_fill
        cell.font = normal
        cell.alignment = Alignment(
            horizontal='center' if col in [1, 5] else 'left',
            vertical='center',
            wrap_text=True
        )

    # Color risk tier cells
    for col in [6, 7]:
        risk_val = values[col - 1]
        if risk_val in tier_fills:
            ws1.cell(excel_row, col).fill = tier_fills[risk_val]
            ws1.cell(excel_row, col).font = tier_fonts[risk_val]
            ws1.cell(excel_row, col).alignment = Alignment(horizontal='center')

    # Color connectivity
    conn_cell = ws1.cell(excel_row, 4)
    if row['connectivity'] == 'No connectivity':
        conn_cell.fill = PatternFill('solid', fgColor='FFCCCC')
        conn_cell.font = Font(bold=True, color='CC0000', size=10)
    else:
        conn_cell.fill = PatternFill('solid', fgColor='CCEECC')
        conn_cell.font = Font(bold=True, color='2D8A4E', size=10)

    ws1.row_dimensions[excel_row].height = 30

# Column widths
col_widths = [6, 35, 18, 18, 10, 12, 12, 45]
for col, width in enumerate(col_widths, 1):
    ws1.column_dimensions[get_column_letter(col)].width = width

# ── Sheet 2 — Detailed Risk Scores ──
ws2 = wb.create_sheet('Detailed Risk Scores')

ws2.merge_cells('A1:G1')
ws2['A1'] = 'Detailed Risk Component Scores'
ws2['A1'].fill = BLUE
ws2['A1'].font = title_font
ws2['A1'].alignment = Alignment(horizontal='center', vertical='center')
ws2.row_dimensions[1].height = 30

headers2 = [
    'School', 'Flood Score', 'Cyclone Score',
    'Vulnerability', 'Hospital (km)', 'HEV Score', 'Risk Tier'
]
for col, h in enumerate(headers2, 1):
    cell = ws2.cell(row=2, column=col, value=h)
    cell.fill = BLUE
    cell.font = bold_white
    cell.alignment = Alignment(horizontal='center')
ws2.row_dimensions[2].height = 22

for i, row in df.iterrows():
    excel_row = i + 3
    row_fill = GRAY if i % 2 == 0 else WHITE
    values2 = [
        row['school_name'],
        row['flood_score'],
        row['cyclone_score'],
        row['vulnerability'],
        row['hospital_km'],
        row['hev_score'],
        row['hev_risk']
    ]
    for col, val in enumerate(values2, 1):
        cell = ws2.cell(row=excel_row, column=col, value=val)
        cell.fill = row_fill
        cell.font = normal
        cell.alignment = Alignment(
            horizontal='center' if col > 1 else 'left',
            vertical='center'
        )
    # Color risk
    ws2.cell(excel_row, 7).fill = tier_fills.get(row['hev_risk'], WHITE)
    ws2.cell(excel_row, 7).font = tier_fonts.get(row['hev_risk'], normal)
    ws2.cell(excel_row, 7).alignment = Alignment(horizontal='center')
    ws2.row_dimensions[excel_row].height = 22

col_widths2 = [35, 12, 14, 14, 14, 12, 12]
for col, width in enumerate(col_widths2, 1):
    ws2.column_dimensions[get_column_letter(col)].width = width

# ── Sheet 3 — Summary Stats ──
ws3 = wb.create_sheet('Summary')

ws3.merge_cells('A1:C1')
ws3['A1'] = 'Executive Summary Statistics'
ws3['A1'].fill = BLUE
ws3['A1'].font = title_font
ws3['A1'].alignment = Alignment(horizontal='center', vertical='center')
ws3.row_dimensions[1].height = 30

stats = [
    ('Total schools analysed', 15, ''),
    ('CRITICAL risk schools', len(df[df.hev_risk == 'CRITICAL']), ''),
    ('HIGH risk schools', len(df[df.hev_risk == 'HIGH']), ''),
    ('MEDIUM risk schools', len(df[df.hev_risk == 'MEDIUM']), ''),
    ('Schools with no connectivity', len(df[df.connectivity == 'No connectivity']), ''),
    ('Schools > 50km from hospital', len(df[df.hospital_km > 50]), ''),
    ('Schools needing urgent action', len(df[df.recommended_action.str.startswith('URGENT')]), ''),
    ('Data sources used', 6, ''),
    ('Years of data coverage', 40, ''),
    ('Analysis date', 'March 2026', ''),
]

for i, (label, value, note) in enumerate(stats):
    row = i + 2
    ws3.cell(row, 1, label).font = Font(bold=True, size=11)
    ws3.cell(row, 2, value).font = Font(size=11, color='065A82', bold=True)
    ws3.cell(row, 1).fill = GRAY if i % 2 == 0 else WHITE
    ws3.cell(row, 2).fill = GRAY if i % 2 == 0 else WHITE
    ws3.row_dimensions[row].height = 22

ws3.column_dimensions['A'].width = 40
ws3.column_dimensions['B'].width = 20

# Save
wb.save('tamil_nadu_school_risk_report.xlsx')
print('✅ Excel report saved!')
print('Sheets: Priority Action List, Detailed Risk Scores, Summary')

✅ Excel report saved!
Sheets: Priority Action List, Detailed Risk Scores, Summary


In [5]:
# Download
from google.colab import files

files.download('tamil_nadu_school_risk_report.xlsx')
print('✅ Excel report downloaded!')

# Summary
print()
print('=== DAY 10 SUMMARY ===')
print(f"CRITICAL schools: {len(df[df.hev_risk == 'CRITICAL'])}")
print(f"Schools needing URGENT action: {len(df[df.recommended_action.str.startswith('URGENT')])}")
print(f"Schools with no connectivity: {len(df[df.connectivity == 'No connectivity'])}")
print(f"Schools > 50km from hospital: {len(df[df.hospital_km > 50])}")
print()
print('Excel sheets created:')
print('  📋 Priority Action List')
print('  📊 Detailed Risk Scores')
print('  📈 Summary Statistics')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Excel report downloaded!

=== DAY 10 SUMMARY ===
CRITICAL schools: 10
Schools needing URGENT action: 8
Schools with no connectivity: 7
Schools > 50km from hospital: 4

Excel sheets created:
  📋 Priority Action List
  📊 Detailed Risk Scores
  📈 Summary Statistics
